# GridCare Project
## Member 1 – Data Engineering

**Student Name:** Dapaah Desmond

**Student ID:** 09902029

### Objective
This notebook focuses on:

- Loading the electricity grid datasets.
- Inspecting and understanding the datasets.
- Cleaning and validating the data.
- Integrating the datasets.
- Preparing the data for graph analysis.

### Datasets

- utilities.csv
- substations.csv
- lines.csv


## 1. Configure Project Paths

Before loading the datasets, I defined the project directories using
`pathlib`. This makes the notebook portable and easier to maintain across
different operating systems. To get future Error raises.

## 2. Loading the datasets

The three datasets are loaded into a Pandas DataFrame
Each DataFrame represents one part of the electricity grid system

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent[1]

# #getting project or folder directory 
# PROJECT_ROOT = Path.cwd().parent
# get the path directory for the folder containing the files
RAW_DATA_DIR = PROJECT_ROOT / "data"/ "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

#loading the files
utilities = pd.read_csv(RAW_DATA_DIR / "utilities.csv")

substations = pd.read_csv(RAW_DATA_DIR / "substations.csv")

lines = pd.read_csv(RAW_DATA_DIR / "lines.csv")

# checking if files have been loaded successfully
print((RAW_DATA_DIR/"utilities.csv").exists())
print((RAW_DATA_DIR / "substations.csv").exists())
print((RAW_DATA_DIR/"lines.csv").exists())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Desmond Dapaah\\Integrated-Systems-Project\\grid_analysis\\data\\raw\\utilities.csv'

## Inspecting the utilities DataFrames

I inspected the utilities DataFrames to understand its structure, quality and limitations
whether there are missing values or any.

In [ ]:
utilities.head()


In [ ]:
utilities.tail()


In [ ]:
utilities.info()


In [ ]:
utilities.describe()


In [ ]:
utilities.isnull().sum()

In [ ]:
utilities.shape

## Inspecting the substations DataFrames

I inspected the substations DataFrames to understand its structure, quality and limitations
whether there are missing values or any.

In [ ]:
substations.head()

In [ ]:
substations.info()

In [ ]:
substations.describe()

In [ ]:
substations.isnull().sum()

In [ ]:
substations.shape

## Inspecting the lines DataFrames

I inspected the substations DataFrames to understand its structure, quality and limitations
whether there are missing values or any.

In [ ]:
lines.head()

In [ ]:
lines.info()

In [ ]:
lines.describe()

In [ ]:
lines.isnull().sum()

In [ ]:
lines.shape

## Data Validation

I am checking whether the data can be trusted enough before merging it to build a network and then analyse.
I will be checking primary key uniqueness, validity of reference key

In [ ]:
# utilities,substations, lines  DataFrame primary key uniqueness

print(f"utilities duplicate Utility ID: {utilities['Utility ID'].duplicated().sum()}")

print(f"substations duplicate Substation ID: {substations['Substation ID'].duplicated().sum()}")

print(f"lines duplicate Line ID: {lines['Line ID'].duplicated().sum()}")
# outputing zero means the primary key is unique

In [ ]:
# Validating foriegn keys, especially from the lines DataFrame since it shows relationship

utility_check = lines['Utility ID'].isin(utilities["Utility ID"])
print(f"{utility_check.value_counts()}")

source_check = lines["Source Substation ID"].isin(substations["Substation ID"])
print(f"{source_check.value_counts()}")

destination_check = lines["Destination Substation ID"].isin(substations["Substation ID"])
print(f"{destination_check.value_counts()}")

## Data Integration
To effectively analyse the network, i need to merge the separate datasets together to form a relationship.
From the three datasets, they are three different relationships.

 - utilities - lines , connected by the Utility ID
 - substations - lines, connected by Source Substation ID
 - substations - lines, connected by Destination Substation ID

To merge these datasets, there are four ways inner, left, right or outer join.
But since I validated that the datasets are cleaned and has no missing values, I can use any.
In this situation I am using the Left join, where every row in the left table is maintained.

To create my relationships, i asked myself " What is the central object we want to analyse", and that electricity grid networks
### Option A: Utilities

 - each row = one utility
 - Too high level

### Option B: Substations

 - ecah row = one substation
 - good for assets analysis

### Option C: Lines

 - one row = one conncetion
 - Excellent for network analysis, since connections build the network

### For this project
I will build two master tables for future scability and to reduce large tables;
1. Lines Utility Table
 - each row is a transmission or distribution line
 - enriched with utility information 
 
2. Lines Master Table
 - First merge Source Substation ID with Substation ID
 - Then merge Destination Substation ID with Substation ID
 - each row is a transmission line from source to destination 
 - enriched with each substation's information and that of utilities information


 

In [ ]:
print(utilities.columns)
print(substations.columns)
print(lines.columns)
lines_with_utilities = pd.merge(lines, utilities , how = "left", on = "Utility ID")
lines_with_utilities.head()

In [ ]:
# Changing the names of the source and destination substations.
#Since they are all substations hence have similar names, so we have to differentiate them

source_substations = substations.rename(
    columns = {
        'Substation ID': 'Source Substation ID',
        'Name' : 'Substation Name',
        'Short Name':'Source Short Name',
        'Region': 'Source Region',
        'Country': ' Source Country',
        'Latitude' : 'Source Latitude',
        'Longitude': 'Source Longitude',
        'Voltage (kV)':'Source Voltage (kV)',
        'Capacity (MVA)': 'Source Capacity (MVA)',
        'Commissioning Year': 'Source Commissioning Year',
        'Type': 'Source Type',
        'Status': 'Source Status'
        }
)

destination_substations = substations.rename(
    columns = {
        'Substation ID': 'Destination Substation ID',
        'Name': 'Substation Name',
        'Short Name':'Destination Short Name',
        'Region': 'Destination Region',
        'Country': ' Destination Country',
        'Latitude' : 'Destination Latitude',
        'Longitude': 'Destination Longitude',
        'Voltage (kV)':'Destination Voltage (kV)',
        'Capacity (MVA)': 'Destination Capacity (MVA)',
        'Commissioning Year': 'Destination Commissioning Year',
        'Type': 'Destination Type',
        'Status': 'Destination Status'
        })

print(source_substations.columns)
print(destination_substations.columns)

In [ ]:
# merging lines_with_utilities and source subastations
lines_with_source = pd.merge(lines_with_utilities, source_substations, how= "left", on="Source Substation ID")
lines_with_source.head()
print(lines_with_source.columns)

In [ ]:
# merging the results from the above  with destination substations to get the whole network table.
lines_with_source_destination = pd.merge(lines_with_source, destination_substations, how= "left", on="Destination Substation ID")
print(lines_with_source_destination.columns)

lines_with_source_destination.head()


In [ ]:
lines_with_source_destination.isnull().sum()

In [ ]:
# renmaning some colums to prevent confusion

lines_with_source_destination = lines_with_source_destination.rename(
    columns = {
        'Name':'Utility Name',
        'Alias' :"Utility Alias",
        'Code':'Utility Code',
        'Type': 'Utility Type',
        'Country': 'Utility Country',
        'Substation Name_x': 'Source Name',
        'Substation Name_y': 'Destination Name',
        'Length (km)': 'Line Length (km)',
        'Voltage (kV)': 'Line Voltage (kV)',
        'Capacity (MVA)': 'Line Capacity (MVA)',
        'Status': 'Line Status'

    }
)

lines_with_source_destination.columns

## Saving Processed Dataset
Saving the ready processed Pandas DataFrame as a csv file.

In [ ]:
# conversion from Pandas DataFrame to csv file
lines_with_source_destination.to_csv("processed_data.csv", index= False)

# checking it was succesful or it exist
Path("processed_data.csv").exists()


In [ ]:
# opening file to verify the conversion was successful

with open("processed_data.csv", "r", encoding="utf-8") as file:
    processed_data = file.read()
    print(processed_data)
